## Today {.smaller}

Last week ended with two measured failures of recurrence: **a gradient that
cannot reach back**, and **one sequential step per token**. One operation fixes
both.

- **Attention**, defined properly. What a query, a key and a value are, and
  what the operation computes.
- **Multi-head attention**, and what a head is for.
- **Positional encoding**, because attention has no idea what order anything
  came in.
- **The block**: what surrounds an attention layer in a real transformer.
- **Encoder, decoder, or both**, and which models are which.

## Labs and quizzes {.smaller}

**Lab 7** was due yesterday. Tokenizing, a vocabulary, padding, masking, and
the measurement showing an unmasked pooled vector is wrong by exactly
$1 - \ell/T$.

**Quiz 8** is at the end of class. It covers the Oct 15 lecture and Lab 7:
sequence data, tokens, embeddings, padding, masking, recurrence and what gating
did. **Nothing from today is on it.** The study guide and formula sheet are on
the course site.

**Lab 8** goes out today, due Wednesday Oct 28.

## What Week 8 already paid for {.smaller}

:::: {.columns}
::: {.column width="50%"}
**Used today, unchanged.** None of this is re-taught.

- tokenization and the vocabulary
- embeddings as a learned lookup table, one row per token
- padding a batch to its longest member
- a mask as a boolean tensor the same shape as the batch
- reading $(N, T, d)$ shapes
:::
::: {.column width="50%"}
**The two failures this week answers.**

The gradient reaching back decayed by a measured 0.53 per step, so a dependency
60 tokens away was unreachable by gradient descent.

And $h_t$ needed $h_{t-1}$, so no amount of hardware made the steps run
together.
:::
::::

# Attention

## What a transformer is {.smaller}

**A transformer is a neural network that relates every position in a sequence
to every other position directly, using attention, with no recurrence.**

[![](images/information-flow.png){.dw68}](images/information-flow.png){target="_blank" .zoom}

Compare the two routes. In a recurrent network, information from the first
token reaches the last by passing through every step in between. In a
transformer, it goes in one hop. Everything else in the architecture is
ordinary: linear layers, a nonlinearity, normalization and residual
connections, all of which you have already seen.

## Why self-attention {.smaller}

The table from *Attention Is All You Need*, 2017. $n$ is the sequence length,
$d$ the representation width, $k$ a convolution's kernel size.

$$\begin{array}{lccc}
\hline
\text{Layer type} & \text{Complexity per layer} &
\text{Sequential operations} & \text{Maximum path length} \\
\hline
\text{Self-attention} & O(n^2 \cdot d) & O(1) & O(1) \\
\text{Recurrent} & O(n \cdot d^2) & O(n) & O(n) \\
\text{Convolutional} & O(k \cdot n \cdot d^2) & O(1) & O(\log_k n) \\
\hline
\end{array}$$

Read the last two columns. **Sequential operations** is what Week 8 measured as
wall clock, and self-attention is $O(1)$ there: every position is computed at
the same time. **Maximum path length** is how far information has to travel,
and it is $O(1)$ too.

The price is the first column. Attention is quadratic in length, and that is
the limit the whole field has been working against ever since.

## What attention is for {.smaller}

:::: {.columns}
::: {.column width="62%"}
The idea underneath attention is **context**. When you read a sentence you
relate some words to others, and which ones depends on the sentence. In "wow,
that rocket ship is moving really fast", everyone knows `fast` describes the
rocket.

Below, `dog` has a strong tie to `barked`, `cats` and `furry`, and a weak one
to `the` and `park`.

[![](images/attention-sentence.png){.dw70}](images/attention-sentence.png){target="_blank" .zoom}
:::
::: {.column width="38%"}
[![](images/attention-overview.png){.dw85}](images/attention-overview.png){target="_blank" .zoom}
:::
::::

**Attention is a way of computing those ties as numbers**, from the tokens
themselves, so the model can learn which ties matter.

## Query, key and value {.smaller}

This is the one new idea in the architecture, and everything else follows from
it. Every token produces three different vectors from its own embedding, each
by its own learned linear layer:

- **query** $q$: what this token is looking for
- **key** $k$: what this token offers to anyone looking
- **value** $v$: what this token actually contributes if attended to

$$\mathbf{Q} = \mathbf{X}\mathbf{W_Q}, \qquad
  \mathbf{K} = \mathbf{X}\mathbf{W_K}, \qquad
  \mathbf{V} = \mathbf{X}\mathbf{W_V}$$

A token's attention to another is the **dot product of its query with that
token's key**: a large dot product means the key matches what the query was
looking for. Those scores become weights, and the output is the weighted sum of
the values.

The library analogy is the standard one and it is good. The query is your
request, the keys are the spine labels, the values are the books.

## Scaled dot product attention {.smaller}

$$\mathbf{A}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) =
  S\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

[![](images/scaled-dot-product.png){.dw50}](images/scaled-dot-product.png){target="_blank" .zoom}

Four steps, and only the third is new. Multiply queries by keys to score every
pair, divide by $\sqrt{d_k}$, softmax each row into weights that sum to 1, then
multiply by the values.

## Why it is divided by the square root {.smaller}

[![](images/fig-scale.png){.dw95}](images/fig-scale.png){target="_blank" .zoom}

A dot product of two $d_k$-dimensional vectors is a sum of $d_k$ terms, so its
spread grows like $\sqrt{d_k}$. At $d_k = 512$ the unscaled softmax puts
**91%** of its weight on a single token, against
**3%** when scaled.

Saturation is not the real cost. **A saturated softmax has almost no gradient**,
so the layer stops learning: the scaled version delivers **6
times** more gradient to its logits. Dividing by $\sqrt{d_k}$ is not cosmetic.

## The attention weight matrix {.smaller}

**The attention matrix is sequence by sequence.** Entry $(i, j)$ is how much
token $i$ attends to token $j$, and every row sums to 1.

[![](images/attention-matrix.png){.dw92}](images/attention-matrix.png){target="_blank" .zoom}

This is the object to picture when anyone says "attention". It is $n \times n$,
which is where the quadratic cost comes from, and it is recomputed from the
content at every layer.

## What attention does to a token {.smaller}

The weights exist to **mix the value vectors**, so a token's output carries its
context rather than only itself. Below, `barked` lets `dog` strongly influence
its feature values, because the query from `barked` matched the key from `dog`.

[![](images/attention-reweights.png){.dw95}](images/attention-reweights.png){target="_blank" .zoom}

The output is the same shape as the input, $(N, T, d)$. What changed is that
every position now holds information from the positions it attended to.

# One attention layer, step by step

## Step 1: project into queries, keys and values {.smaller}

[![](images/encoder-step-1.png){.dw75}](images/encoder-step-1.png){target="_blank" .zoom}

Three matrix multiplies on the same input. $\mathbf{X}$ is $(s, m)$ for $s$
tokens of width $m$, and each projection takes it to $(s, e)$. **Click any
diagram in the handout to open it full size**; these are drawn to be zoomed.

## Step 2: score every pair, then soften {.smaller}

[![](images/encoder-step-2.png){.dw100}](images/encoder-step-2.png){target="_blank" .zoom}

$\mathbf{Q}\mathbf{K}^\top$ is $(s, s)$: every token against every token.
Divide by $\sqrt{d_k}$, add the padding mask, and softmax each row. The mask
adds $-\infty$ at pad positions, and $e^{-\infty} = 0$, so those columns
contribute nothing.

## Step 3: take the weighted sum of the values {.smaller}

[![](images/encoder-step-3.png){.dw75}](images/encoder-step-3.png){target="_blank" .zoom}

The weights multiply $\mathbf{V}$, giving $(s, e)$ again. Every output row is a
blend of value vectors, weighted by how much that token attended to each.

## Attention in code {.smaller .codetight}

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(6600)
N, T, d = 2, 6, 32                       # batch, tokens, model width

x = torch.randn(N, T, d)
Wq, Wk, Wv = (torch.nn.Linear(d, d, bias=False) for _ in range(3))
Q, K, V = Wq(x), Wk(x), Wv(x)

scores = (Q @ K.transpose(-2, -1)) / d ** 0.5     # (N, T, T)
weights = scores.softmax(dim=-1)
out = weights @ V                                  # (N, T, d)

print(f"x       {tuple(x.shape)}")
print(f"scores  {tuple(scores.shape)}   <- the n-by-n matrix, per example")
print(f"weights rows sum to {weights.sum(-1).min():.4f}")
print(f"out     {tuple(out.shape)}   <- same shape as the input")
print()
print("torch does the whole thing in one call:")
ref = F.scaled_dot_product_attention(Q, K, V)
print(f"  matches the manual version: {torch.allclose(out, ref, atol=1e-5)}")

x       (2, 6, 32)
scores  (2, 6, 6)   <- the n-by-n matrix, per example
weights rows sum to 1.0000
out     (2, 6, 32)   <- same shape as the input

torch does the whole thing in one call:
  matches the manual version: True


# Multi-head attention

## One head is not enough {.smaller}

A single attention layer produces **one** weight matrix, so it can express one
notion of relatedness per layer. Language has several at once: who did what to
whom, which noun a pronoun refers to, which words are the same phrase.

**Multi-head attention runs $h$ attention operations in parallel on different
learned projections of the same input, then concatenates the results.**

[![](images/multi-head.png){.dw65}](images/multi-head.png){target="_blank" .zoom}

## What the heads actually do {.smaller}

[![](images/fig-heads.png){.dw100}](images/fig-heads.png){target="_blank" .zoom}

Every head in one real layer of DistilBERT, on one sentence. They are visibly
not doing the same job: **head 4 puts 62% of its
weight on the previous token**, and head 6 puts 88% on
the separator, which is this model's way of attending to nothing in particular.

## Heads are free {.smaller}

The projections shrink to match. With $d = 768$ and $h = 12$, each head
works in **64** dimensions, so the four projection matrices cost
**2,359,296** parameters either way.

$$\text{MultiHead}(\mathbf{X}) =
  \big[\, \text{head}_1 ; \dots ; \text{head}_h \,\big]\,\mathbf{W_O}$$

So you get 12 different relationships for the price of one. That is why
every transformer uses many heads, and why the number of heads is usually a
round divisor of the width.

## Multi-head attention in code {.smaller .codetight}

In [2]:
import torch
import torch.nn as nn

torch.manual_seed(6600)
N, T, d, h = 2, 6, 768, 12
mha = nn.MultiheadAttention(d, num_heads=h, batch_first=True)
x = torch.randn(N, T, d)

out, w = mha(x, x, x, need_weights=True, average_attn_weights=False)
print(f"out     {tuple(out.shape)}")
print(f"weights {tuple(w.shape)}   <- (batch, heads, T, T)")
print(f"each head works in {d // h} dimensions")
print()
print("self-attention means the same tensor is query, key and value:")
print("   mha(x, x, x)     <- self-attention")
print("   mha(y, x, x)     <- cross-attention, query from y, keys from x")

out     (2, 6, 768)
weights (2, 12, 6, 6)   <- (batch, heads, T, T)
each head works in 64 dimensions

self-attention means the same tensor is query, key and value:
   mha(x, x, x)     <- self-attention
   mha(y, x, x)     <- cross-attention, query from y, keys from x


# Position

## Attention has no sense of order {.smaller}

:::: {.columns}
::: {.column width="58%"}
Look again at what attention computes. Every output is a weighted sum over the
whole sequence, and **nothing in it depends on where a token sits**. Shuffle the
input and the output is shuffled the same way, with identical values.

That is fatal for language. "The dog bit the man" and "the man bit the dog" are
the same bag of tokens.

**Positional encoding adds a vector to each token's embedding that depends only
on its position**, so position becomes part of the content attention sees.
:::
::: {.column width="42%"}
[![](images/posenc-overview.png){.dw80}](images/posenc-overview.png){target="_blank" .zoom}
:::
::::

## Sinusoidal positional encoding {.smaller}

$$P(k, 2i) = \sin\!\left(\frac{k}{n^{2i/d}}\right), \qquad
  P(k, 2i+1) = \cos\!\left(\frac{k}{n^{2i/d}}\right)$$

[![](images/fig-posenc.png){.dw95}](images/fig-posenc.png){target="_blank" .zoom}

$k$ is the position, $i$ indexes the dimension pair, and $n = 10{,}000$ is the
authors' choice. Each dimension is a wave of a different wavelength, so every
position gets a distinct pattern.

The right panel is why this works at all. Similarity between two encodings
depends on **the gap between them**, not on where they are: neighbouring
positions sit at 0.97, positions 20 apart at 0.61, from anywhere in
the sequence.

## Learned positional encoding {.smaller}

[![](images/learned-posenc.png){.dw75}](images/learned-posenc.png){target="_blank" .zoom}

The alternative is to stop deriving it and just learn it: one row per position,
trained like any other parameter. That is what BERT, GPT and ViT all do.

The trade is extrapolation. Sinusoidal encoding is defined for any position, so
a longer sequence than you trained on still gets sensible values. A learned
table has a fixed number of rows and **position 2,049 of a 2,048 row table does
not exist**. Week 10 is largely about fixing that.

# Masking

## Two masks, for two different reasons {.smaller}

:::: {.columns}
::: {.column width="56%"}
**A padding mask stops attention from reading pad tokens.** It is Week 8's mask,
applied inside attention rather than at pooling, and it is needed in both the
encoder and the decoder.

**A causal mask stops a position from attending to anything after it.** It
exists only in the decoder, and only because of how generation works: at
inference the model has not produced the later tokens yet, so training must not
let it see them either.

Both are implemented the same way: add $-\infty$ before the softmax.
:::
::: {.column width="44%"}
[![](images/masking-overview.png){.dw75}](images/masking-overview.png){target="_blank" .zoom}
:::
::::

## The causal mask, applied {.smaller}

[![](images/fig-causal.png){.dw100}](images/fig-causal.png){target="_blank" .zoom}

The left panel is a real attention matrix with no mask: every token attends
everywhere. The middle is the mask, keep on and below the diagonal. The right
is the result.

Two things to notice. The first row is **1.00** on itself, because the
first token has nothing before it to look at. And every row still sums to
**1.00**, because the softmax runs after the mask, not before.

## Masking summary {.smaller}

[![](images/masking-summary.png){.dw65}](images/masking-summary.png){target="_blank" .zoom}

Where each mask applies, in one picture. The encoder gets a padding mask. The
decoder gets a padding mask and a causal mask, combined by adding both. The
cross-attention layer masks the encoder's padding, because that is whose keys
it is reading.

# The block around the attention

## What surrounds an attention layer {.smaller}

Attention is one sublayer. A transformer block has two, each wrapped the same
way:

$$\mathbf{x} \leftarrow \text{LayerNorm}\big(\mathbf{x} +
  \text{Attention}(\mathbf{x})\big), \qquad
  \mathbf{x} \leftarrow \text{LayerNorm}\big(\mathbf{x} +
  \text{FFN}(\mathbf{x})\big)$$

- **The residual connection** is Week 6's, unchanged, and for the same reason:
  it gives the gradient a path with derivative 1 so you can stack many blocks.
- **The feed forward network** is a two-layer MLP applied to each position
  independently, usually widening to $4d$ and back. Attention mixes across
  positions; the FFN is where per-position computation happens.
- **Layer normalization** is the part that is not Week 6's.

## Why LayerNorm and not BatchNorm {.smaller}

[![](images/fig-norm.png){.dw95}](images/fig-norm.png){target="_blank" .zoom}

The same token, every time. Only the amount of padding around it changes.

**BatchNorm normalizes over the batch**, so its answer for this token depends on
the other examples, including how much of the batch is padding: the output
moves by up to **3.11**. **LayerNorm normalizes over the features**
of each token on its own, so the output moves by **0.00**. Exactly
zero, not approximately.

That is the whole reason sequence models normalize over features. A batch of
sentences is ragged, so batch statistics are not a property of the data.

## Layer normalization, in place {.smaller}

[![](images/layer-norm.png){.dw85}](images/layer-norm.png){target="_blank" .zoom}

Per token, subtract that token's mean and divide by its standard deviation,
then scale and shift by learned parameters. `nn.LayerNorm(d)` does it, and in
PyTorch's `TransformerEncoderLayer` it is `norm_first=False` by default, which
puts it after the residual add. Modern models mostly use `norm_first=True`,
which is more stable to train deep.

## A block in code {.smaller .codetight}

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(6600)
d, h, T, N = 256, 8, 10, 4

block = nn.TransformerEncoderLayer(
    d_model=d, nhead=h, dim_feedforward=4 * d,
    batch_first=True, norm_first=True)

x = torch.randn(N, T, d)
pad_mask = torch.zeros(N, T, dtype=torch.bool)
pad_mask[2, 7:] = True                       # example 2 is 7 tokens, then pad

out = block(x, src_key_padding_mask=pad_mask)
print(f"in  {tuple(x.shape)}  ->  out {tuple(out.shape)}   (shape preserved)")
print(f"parameters in one block: {sum(p.numel() for p in block.parameters()):,}")
print()
att = sum(p.numel() for n, p in block.named_parameters() if "self_attn" in n)
ffn = sum(p.numel() for n, p in block.named_parameters() if "linear" in n)
print(f"  attention {att:>9,}")
print(f"  feed fwd  {ffn:>9,}   <- the FFN is the bigger half")

in  (4, 10, 256)  ->  out (4, 10, 256)   (shape preserved)
parameters in one block: 789,760

  attention   263,168
  feed fwd    525,568   <- the FFN is the bigger half


# Decoder and architectures

## The decoder block {.smaller}

:::: {.columns}
::: {.column width="50%"}
A decoder block is an encoder block plus two changes.

**Its self-attention is causally masked**, so position $t$ sees only positions
up to $t$.

**It has a third sublayer, cross-attention**, between the self-attention and the
feed forward network.
:::
::: {.column width="50%"}
[![](images/decoder-notation.png){.dw85}](images/decoder-notation.png){target="_blank" .zoom}
:::
::::

[![](images/decoder-step-3.png){.dw40}](images/decoder-step-3.png){target="_blank" .zoom}

## Cross-attention {.smaller}

**In self-attention, the queries, keys and values all come from the same
sequence. In cross-attention, the queries come from the decoder and the keys
and values come from the encoder.**

[![](images/cross-attention.png){.dw92}](images/cross-attention.png){target="_blank" .zoom}

That single change is what let translation stop squeezing a sentence into one
vector. Each decoder position issues a query and reads the whole input
directly, which is precisely the bottleneck Week 8 ended on.

## Encoder, decoder, or both {.smaller}

The original paper used both halves. Either works alone, and which half you
keep decides what the model is for.

| architecture | sees | example | good at |
|---|---|---|---|
| **encoder only** | the whole input at once | BERT | classification, named entity recognition, extractive QA |
| **decoder only** | everything before this token | GPT | generating text |
| **encoder-decoder** | input fully, output causally | T5 | translation, summarization |

The pattern worth remembering: **bidirectional context is for understanding, and
causal masking is for generating.** You cannot generate with bidirectional
attention, because the thing you would attend to does not exist yet.

## A decoder-only model, training and generating {.smaller}

[![](images/gpt-teacher-forcing.png){.dw50}](images/gpt-teacher-forcing.png){target="_blank" .zoom}

This is Week 8's teacher forcing, now with the mechanism filled in. Training
feeds the whole true sequence at once and predicts every next token in
parallel, which is possible only because the causal mask stops each position
cheating. Generation still runs one token per forward pass.

**The asymmetry is the thing.** Training is parallel over positions; generation
cannot be. Week 10 is largely about making that second half cheap.

# Wrap-up

## The whole architecture {.smaller}

:::: {.columns}
::: {.column width="30%"}
[![](images/full-transformer.png){.dw100}](images/full-transformer.png){target="_blank" .zoom}
:::
::: {.column width="70%"}
Every part of this is now defined. Reading it from the bottom:

- token embeddings, from Week 8, plus a positional encoding
- $N$ encoder blocks: self-attention with a padding mask, then a feed forward
  network, each wrapped in a residual and a layer norm
- $N$ decoder blocks: the same, plus a causal mask on the self-attention and a
  cross-attention sublayer that reads the encoder
- a linear layer to vocabulary size and a softmax

**Open this one full size in the handout.** It is drawn to be read slowly, and
it is the single most useful page to keep beside you for the rest of the
course.
:::
::::

## Go back and re-read two slides {.smaller}

:::: {.columns}
::: {.column width="46%"}
[![](images/vit-paper-diagram.png){.dw100}](images/vit-paper-diagram.png){target="_blank" .zoom}
:::
::: {.column width="54%"}
Week 7 used the words **transformer**, **attention**, **encoder-decoder** and
**query** without defining any of them, and said to come back.

This is that moment. A **ViT** cuts an image into patches, projects each one,
adds a learned position embedding, and runs a **standard transformer encoder**:
the same block from the slides above, with no change.

**DETR** uses an encoder-decoder, and its 100 "object queries" are literally
queries in the sense defined today: 100 learned vectors that each attend to the
image and come back with one object or nothing.
:::
::::

## Wrap-up {.smaller}

- **Attention** gives every position a direct, one-hop route to every other,
  which is what fixes both of recurrence's failures.
- A **query** asks, a **key** advertises, a **value** is what gets delivered.
  The attention weight is a query dotted with a key.
- The $\sqrt{d_k}$ is not cosmetic: without it the softmax saturates and stops
  passing gradient.
- **Many heads** cost the same as one and learn visibly different relationships.
- **Positional encoding** is needed because attention is order-blind.
- **Padding masks** go in both halves; the **causal mask** only in the decoder.
- A block is attention, a feed forward network, residuals, and **LayerNorm**,
  which unlike BatchNorm does not care how much of the batch is padding.
- Encoder for understanding, decoder for generating, both for sequence to
  sequence.

## Supplemental content {.smaller}

Optional. Prof. Hickman's [centralized lecture
content](https://jfh.georgetown.domains/centralized-lecture-content/) covers
this material with different examples:

- [Transformers](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/transformers-introduction/notes.html)
- [Vision transformers](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/computer-vision/vision-transformers/introduction-to-vision-transformers/notes.html)
- [Large language models](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/transformers/llm/notes.html), which is next week

Also optional, and the primary source is unusually readable: Vaswani et al.,
[Attention Is All You Need](https://arxiv.org/abs/1706.03762) (2017). Jay
Alammar's [Illustrated
Transformer](https://jalammar.github.io/illustrated-transformer/) is the best
free explainer on the internet and covers exactly these slides.

## Lab 8 and Quiz 8 {.smaller}

**Lab 8** is out today, due Wednesday Oct 28. You build a transformer encoder
block from `nn.Linear` and `softmax` alone, check it against
`nn.TransformerEncoderLayer`, and then break it three ways on purpose: drop the
scale, drop the positional encoding, and drop the causal mask.

**Quiz 8** is now. Scope is the Oct 15 lecture and Lab 7 only. A formula sheet
comes with it.

Next week is **Large Language Models**: what had to change between the 2017
architecture on these slides and a model you can hold a conversation with.